# FORESEE: DarkPhoton

This notebook estimates the sensitivity of forward LHC experiments to a kinetic-mixing **dark photon** $A'$. It builds the model from `Models/DarkPhoton/build.py`, then walks through the standard FORESEE outputs -- the LLP spectrum at a benchmark point, the production rate versus mass, and the sensitivity reach across detectors -- and can optionally write HepMC event files. Every step is driven through `src/utils/routines.py`, so the notebook is a thin, model-specific wrapper you can run top to bottom without editing.

## 1. Load Libraries

In [ ]:
# The shared plotting routines live in src/utils. Put the library root (the
# folder containing src/) on the path, then import it; this works from any
# notebook location (Examples/ or Models/<Name>/).
import os, sys
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "src")):
    root = os.path.dirname(root)
sys.path.insert(0, root)
from src.utils import routines

## 2. Specifying the Model

The dark photon $A′$ is a new massive vector boson, which could, for example, be the gauge boson of a broken abelian gauge group in a hidden sector. Since the dark photon has the same quantum numbers as the photon of the SM, the two states can kinetically mix. This effectively introduces couplings of the dark photon to all charged SM fermions. The phenomenology of the dark photon can then be described by the following Lagrangian

\begin{equation}
 \mathcal{L} = \frac{1}{2} \textcolor{red}{m_{A'}}^2 A'^2  - \textcolor{red}{\epsilon} e \sum \bar f \gamma^\mu f A'_\mu
\end{equation}

with the dark photon mass $\textcolor{red}{m_{A'}}$ and the kinetic mixing parameter $\textcolor{red}{\epsilon}$ as free parameters.

### A. Model parameters

The block below shows the builder's default kwargs; override any as needed. See `Models/DarkPhoton/build.py` for the full signature.

In [ ]:
# --- DarkPhoton (kinetic-mixing dark photon) ---
MODEL_NAME = "DarkPhoton"
MODEL_PARAMS = dict(
    energy="14",                                       # "13.6", "14", "27", "100"
    nsample_2body=100,                                 # sampling points per channel
    generators_light=["EPOSLHC", "SIBYLL", "QGSJET"],
    generators_heavy=None,                             # no heavy-meson channels
    brem_configurations=["Brem_QRA_L1.5", "Brem_QRA_L1.0", "Brem_QRA_L2.0"],  # QRA pt-cut envelope
)


### B. Load model and attach presets

`routines.load(MODEL_NAME, MODEL_PARAMS)` builds the model, calls `set_model`, and attaches its `build_presets()`. It prints a summary of the loaded defaults and returns two objects:

- `foresee` &mdash; the `Foresee` instance; the configured model is reachable as `foresee.model`
- `presets` &mdash; everything model-specific (benchmark, scan grids, detectors, plot styling), straight from the model's `build_presets()`

It uses the full preset mass grid by default; pass `thin=N` to subsample every Nth mass for faster tutorial runs.

In [ ]:
# Load the model, attach its presets, and return (foresee, presets); the model
# is reachable as foresee.model. Uses the full mass grid by default; pass
# thin=N to subsample every Nth mass for faster tutorial runs.
foresee, presets = routines.load(MODEL_NAME, MODEL_PARAMS, thin = 2)

mass, coupling = presets["benchmark"]["mass"], presets["benchmark"]["coupling"]
masses, couplings = presets["grid"]["masses"], presets["grid"]["couplings"]
detectors = presets["detectors"]


In [ ]:
# Add a custom detector defined explicitly to the scan. The inline "energy" picks the
# beam the model is rebuilt at, so no src/utils/detectors.py registry entry is
# needed; the remaining keys are the geometry handed to foresee.set_detector.
detectors += [{
    "label": "FASER_custom",
    "energy": "13.6",                              # beam energy (inline, no registry lookup)
    "distance": 480, "length": 1.5, "luminosity": 250,
    "selection": "np.sqrt(x.x**2 + x.y**2) < 0.1",
}]

### C. Customizing the plots

**`presets` holds the plot styling.** To restyle a figure, edit the presets *before* calling the matching routine. `plot_production` and `get_sens` hands `presets["production_plot"]` / `["reach_plot"]` straight into FORESEE's plotting calls, so any key those accept (`xlims`, `ylims`, `xlabel`/`ylabel`, `title`, `legendloc`, `figsize`, ...) can be set here.

In [ ]:
presets["production_plot"]["title"] = "Tutorial Production Plot"

## LLP spectrum at the benchmark point

`routines.plot_spectrum(...)` plots the angle-momentum spectrum of the LLP at a single **(mass, coupling)** point -- here the benchmark point -- by calling `foresee.get_llp_spectrum(..., do_plot=True)`, summing over every production channel. The figure is saved under `figures/DarkPhoton/`.

In [ ]:
routines.plot_spectrum(
    foresee,
    mass=mass,
    coupling=1,                              
)

## Production rate vs mass

`routines.plot_production(...)` plots the production rate ($\sigma/\varepsilon^2$) versus mass, one curve per production channel, at the model's beam energy. It works off the cached LLP spectra, so the cell below first calls `routines.cache_spectra(foresee, masses)` to precompute any spectra still missing for the mass grid (masses already on disk are skipped). `plot_production` then draws the channels listed in `presets['production_channels']`, styled by `presets['production_plot']`, optionally with a branching-ratio sub-panel, and saves the figure under `figures/DarkPhoton/`.

`plot_production` calls `cache_spectra` itself, so the explicit call only makes the caching step visible (and lets you cache once before re-plotting with different styling).

In [ ]:
# Precompute (and cache) the LLP spectra across the mass grid, then plot. This
# call is optional -- plot_production caches internally -- but makes the step
# explicit. Masses already cached on disk are skipped.
routines.cache_spectra(foresee, masses)

routines.plot_production(
    foresee, presets,
    masses=masses,                           # defaults to the presets grid
)

## Sensitivity reach

`routines.get_sens(...)` scans the **mass x coupling** grid for every detector in `detectors`, producing one reach curve per **(detector, production configuration)**. Detectors are grouped by beam energy and the model is rebuilt/cached per energy. Each curve is cached to `model/results/` as `<energy>_<detector>_<label>.npy`: a detector that already has a result file is read straight from disk, and only the missing ones are scanned live. The figure overlays the existing `bounds`/`projections`/`lines` from `presets` (drawing only the first, nominal production configuration of each detector) and is saved under `figures/DarkPhoton/`.

In [ ]:
presets["reach_plot"]["legendloc"] = (1.02,.7)
routines.get_sens(
    foresee, presets, MODEL_PARAMS,
    detectors= detectors,          
    masses= masses,        
    couplings= couplings,  
    modes=None,                              # None uses the model's full production set
    labels=None,                             # None -> numeric column indices, one per config
    plot=True,                               
)

## Generate HepMC event files (optional)

`routines.gen_hepmc(...)` loops the scan grid and calls `foresee.write_events` to write one HepMC file per **(detector, mass, coupling)** into `model/events/`. Detectors are grouped by beam energy and the model is rebuilt/cached per energy, just like the reach scan.

In [ ]:
# One HepMC file per detector x mass x coupling, into model/events/. Limited
# here to the benchmark point; the masses=/couplings= grids default to the full
# presets grid (one file per point). See routines.gen_hepmc for all options.
routines.gen_hepmc(
    foresee, presets, MODEL_PARAMS,
    numberevent=100,                              # unweighted events sampled per file
    masses=[mass],        # defaults to the presets grid
    couplings=[coupling], # defaults to the presets grid
    detectors= detectors,               # defaults to the presets detectors
    modes=None,                                   # None uses the model's full production set
    labels=None,                                  # None -> numeric column indices, one per config
    nsample=1,                                    # sampling passed to write_events
    filetype="hepmc",                             # output file type
)